# Does the halfway leader usually win the Bundesliga? The notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/herbstmeister-usually-champion/story_008_herbstmeister.ipynb)

This R notebook recomputes every statistic in
[Does the halfway leader usually win the Bundesliga?](https://pointzerofive.net/2026/10/herbstmeister-usually-champion),
story #008 on *pointzerofive*, from the result of every Bundesliga match since 1963. Colab opens it
with the R runtime; run it top to bottom (`Runtime → Run all`).

Every number is checked against the published results file with `stopifnot`, so the notebook
stops at the first number that does not match. Requires base R and `jsonlite` (preinstalled in
Colab). The pipeline that wrote the dataset is `herbstmeister_pipeline.R`; its console output,
with R and package versions, is `herbstmeister-usually-champion.output.txt`.

In [1]:
library(jsonlite)

BASE <- "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/herbstmeister-usually-champion/"
REPO <- ".."

# A file of this story: next to the notebook (pointzerofive-data), in the website's repository,
# else downloaded from pointzerofive-data.
fetch_path <- function(rel) {
  for (local in c(basename(rel), file.path(REPO, rel))) if (file.exists(local)) return(local)
  paste0(BASE, basename(rel))
}

near <- function(a, b, tol = 1e-6) stopifnot(abs(a - b) <= tol)

read_csv_utf8 <- function(path) {
  read.csv(path, encoding = "UTF-8", stringsAsFactors = FALSE, check.names = FALSE)
}

m <- rbind(read_csv_utf8(fetch_path("notebooks/raw/bundesliga/matches_1963_2010.csv")),
           read_csv_utf8(fetch_path("notebooks/raw/bundesliga/matches.csv")))
ref <- read_csv_utf8(fetch_path("notebooks/raw/bundesliga/reference_lists.csv"))
published <- fromJSON(fetch_path("data/herbstmeister-usually-champion.json"), simplifyVector = FALSE)
m$start <- as.integer(substr(m$season, 1, 4))
m$date <- as.Date(m$date)
cat(sprintf("%d matches, %d seasons (%d to %d)\n", nrow(m), length(unique(m$start)),
            min(m$start), max(m$start)))
stopifnot(nrow(m) == published$summary$matches)

19220 matches, 63 seasons (1963 to 2025)


## 1. The matches

`matches_1963_2010.csv` comes from football.csv (CC0 1.0), an export of openfootball;
`matches.csv` (2010/11 to 2025/26) from openfootball itself (CC0 1.0). Every club plays every other
club twice, so a season with *c* clubs has *c*(*c* − 1) matches and 2(*c* − 1) matchdays: 16 clubs in
1963/64 and 1964/65, 20 clubs in 1991/92, 18 otherwise.

In [2]:
seasons <- sort(unique(m$start))
clubs <- sapply(seasons, function(s) length(unique(unlist(m[m$start == s, c("home", "away")]))))
for (i in seq_along(seasons)) {
  d <- m[m$start == seasons[i], ]
  stopifnot(nrow(d) == clubs[i] * (clubs[i] - 1), max(d$matchday) == 2 * (clubs[i] - 1))
}
print(table(clubs))

clubs
16 18 20 
 2 60  1 


## 2. The tables

Points as awarded at the time: two for a win until 1994/95, three from 1995/96. Ties on points are
broken by goal quotient until 1968/69, then by goal difference, then by goals scored. The
**halfway leader** (Herbstmeister) tops the table of all first-half matches (matchdays 1 to
*c* − 1, postponed matches included); the winter-break table counts every match up to 31 December
of the season's first year. The **champion** tops the final table.

In [3]:
standings <- function(d, win_pts, quotient) {
  team <- c(d$home, d$away)
  gf <- c(d$home_goals, d$away_goals)
  ga <- c(d$away_goals, d$home_goals)
  pts <- ifelse(gf > ga, win_pts, ifelse(gf == ga, 1, 0))
  t <- data.frame(team = sort(unique(team), method = "radix"))
  f <- factor(team, levels = t$team)
  t$pts <- as.vector(tapply(pts, f, sum))
  t$gf <- as.vector(tapply(gf, f, sum))
  t$ga <- as.vector(tapply(ga, f, sum))
  t$gd <- t$gf - t$ga
  t$gq <- ifelse(t$ga == 0, Inf, t$gf / t$ga)
  o <- if (quotient) order(-t$pts, -t$gq, t$team, method = "radix") else
    order(-t$pts, -t$gd, -t$gf, t$team, method = "radix")
  t[o, ]
}

rows <- do.call(rbind, lapply(seq_along(seasons), function(i) {
  s <- seasons[i]
  d <- m[m$start == s, ]
  wp <- if (s >= 1995) 3 else 2
  q <- s <= 1968
  final <- standings(d, wp, q)
  do.call(rbind, lapply(c("first_half", "winter_break"), function(tab) {
    part <- if (tab == "first_half") d[d$matchday <= clubs[i] - 1, ] else
      d[d$date <= as.Date(sprintf("%d-12-31", s)), ]
    ht <- standings(part, wp, q)
    ht3 <- standings(part, 3, q)
    leader <- ht$team[1]
    data.frame(start = s, table = tab, leader = leader, champion = final$team[1],
               level = ht$pts[1] == ht$pts[2], place = match(leader, final$team),
               lead3 = ht3$pts[ht3$team == leader] - max(ht3$pts[ht3$team != leader]),
               y = as.integer(leader == final$team[1]), bayern = leader == "Bayern München")
  }))
}))
fh <- rows[rows$table == "first_half", ]
wb <- rows[rows$table == "winter_break", ]

# against the published rows and the two reference lists
for (i in seq_len(nrow(fh))) {
  p <- published$seasons[[i]]
  stopifnot(fh$place[i] == p$place, fh$y[i] == p$y, fh$lead3[i] == p$lead3,
            fh$level[i] == p$levelOnPoints, fh$bayern[i] == p$bayern, wb$y[i] == p$winterY)
}
stopifnot(all(fh$y == ref$herbstmeister_champion_de_wikipedia))
cat(sprintf("%d seasons; halfway leader became champion in %d (Wikipedia: %d)\n", nrow(fh),
            sum(fh$y), sum(ref$herbstmeister_champion_de_wikipedia)))

63 seasons; halfway leader became champion in 44 (Wikipedia: 44)


## 3. First results (Fig. 1)

How often the halfway leader won the title, for Bayern and for everyone else, and where the
others finished. Up to 2019/20 the count can be compared with the DSFS (38 of 57).

In [4]:
s <- published$summary
stopifnot(sum(fh$y) == s$all$champion, nrow(fh) == s$all$n,
          sum(fh$y[fh$bayern]) == s$bayern$champion, sum(fh$bayern) == s$bayern$n,
          sum(fh$y[!fh$bayern]) == s$others$champion, sum(!fh$bayern) == s$others$n,
          sum(fh$level) == s$levelOnPoints$n, sum(fh$leader != wb$leader) == s$winterDiffers)
cat(sprintf("all: %d of %d; Bayern: %d of %d; others: %d of %d\n", sum(fh$y), nrow(fh),
            sum(fh$y[fh$bayern]), sum(fh$bayern), sum(fh$y[!fh$bayern]), sum(!fh$bayern)))
cat(sprintf("up to 2019/20: %d of %d\n", sum(fh$y[fh$start <= 2019]), sum(fh$start <= 2019)))
miss <- fh[fh$y == 0, c("start", "leader", "place", "champion")]
print(miss, row.names = FALSE)
cat(sprintf("missed the title but finished second: %d of %d\n", sum(miss$place == 2), nrow(miss)))

all: 44 of 63; Bayern: 25 of 28; others: 19 of 35


up to 2019/20: 38 of 57


 start               leader place             champion
  1970       Bayern München     2 Bor. Mönchengladbach
  1971        FC Schalke 04     2       Bayern München
  1978 1. FC Kaiserslautern     3         Hamburger SV
  1980         Hamburger SV     2       Bayern München
  1981           1. FC Köln     2         Hamburger SV
  1985        Werder Bremen     2       Bayern München
  1986         Hamburger SV     2       Bayern München
  1990        Werder Bremen     3 1. FC Kaiserslautern
  1991  Eintracht Frankfurt     3        VfB Stuttgart
  1992       Bayern München     2        Werder Bremen
  1993  Eintracht Frankfurt     5       Bayern München
  2000        FC Schalke 04     2       Bayern München
  2001  Bayer 04 Leverkusen     2    Borussia Dortmund
  2006        Werder Bremen     3        VfB Stuttgart
  2008  TSG 1899 Hoffenheim     7        VfL Wolfsburg
  2009  Bayer 04 Leverkusen     4       Bayern München
  2011       Bayern München     2    Borussia Dortmund
  2018    

missed the title but finished second: 11 of 19


## 4. The test: logistic regression on a constant

y = 1 if the halfway leader became champion. `glm(y ~ 1, family = binomial)` estimates the
log-odds of the share; the Wald z-test of the constant against zero is the test of a share of one
half. The 95% confidence interval is the constant ± 1.96 standard errors, converted with the
logistic function. Effect = share − 0.5, in percentage points.

In [5]:
logit_test <- function(y) {
  fit <- glm(y ~ 1, family = binomial)
  co <- summary(fit)$coefficients
  b <- co[1, 1]; se <- co[1, 2]; z <- qnorm(0.975)
  list(effect = 100 * (plogis(b) - 0.5), ci_low = 100 * (plogis(b - z * se) - 0.5),
       ci_high = 100 * (plogis(b + z * se) - 0.5), p = co[1, 4], n = length(y), k = sum(y),
       b = b, se = se, fit = fit)
}
main <- logit_test(fh$y)
pub <- published$specs[[1]]
stopifnot(identical(unlist(pub$choices), c(seasons = "all", table = "first_half", ties = "official")))
near(main$effect, pub$effect); near(main$ci_low, pub$ci_low); near(main$ci_high, pub$ci_high)
near(main$p, pub$p, 1e-12)
cat(sprintf("share %.4f, effect %+.2f percentage points, 95%% CI [%+.2f, %+.2f], p = %.4f\n",
            plogis(main$b), main$effect, main$ci_low, main$ci_high, main$p))
print(summary(main$fit)$coefficients)
est <- published$estimation$rows
near(est[[1]]$values[[1]], main$b); near(est[[1]]$values[[2]], main$se)
chi <- chisq.test(c(main$k, main$n - main$k), p = c(0.5, 0.5))
cat(sprintf("chi-squared test against 50:50: p = %.4f (same verdict)\n", chi$p.value))

share 0.6984, effect +19.84 percentage points, 95% CI [+7.49, +29.86], p = 0.0022


             Estimate Std. Error  z value    Pr(>|z|)
(Intercept) 0.8397507  0.2745155 3.059028 0.002220566


chi-squared test against 50:50: p = 0.0016 (same verdict)


## 5. The size of the lead (Fig. 2)

The halfway leader's lead over the best other club, recounted with three points per win for all
seasons. The logistic curve on the lead is descriptive only.

In [6]:
by_lead <- function(lo, hi) {
  r <- fh[fh$lead3 >= lo & fh$lead3 <= hi, ]
  c(k = sum(r$y), n = nrow(r))
}
print(rbind(`6 or more` = by_lead(6, 99), `2 or less` = by_lead(0, 2)))
curve <- glm(y ~ lead3, family = binomial, data = fh)
near(coef(curve)[[1]], published$leadCurve$intercept); near(coef(curve)[[2]], published$leadCurve$slope)
cat(sprintf("curve: %.0f%% with no lead, %.0f%% with six points\n",
            100 * plogis(coef(curve)[[1]]), 100 * plogis(coef(curve)[[1]] + 6 * coef(curve)[[2]])))

           k  n
6 or more 15 17
2 or less 15 29


curve: 46% with no lead, 85% with six points


## 6. All 12 calculation paths

Seasons: all, without the seasons Bayern led at halfway, or only since 1995/96. Table: after the
first half or on 31 December. Ties: as the table shows them, or only leaders ahead on points.

In [7]:
grid <- expand.grid(ties = c("official", "clear_lead"), table = c("first_half", "winter_break"),
                    seasons = c("all", "without_bayern", "three_points"),
                    stringsAsFactors = FALSE)[, 3:1]
out <- list()
for (i in seq_len(nrow(grid))) {
  ch <- grid[i, ]
  d <- rows[rows$table == ch$table, ]
  if (ch$seasons == "without_bayern") d <- d[!d$bayern, ]
  if (ch$seasons == "three_points") d <- d[d$start >= 1995, ]
  if (ch$ties == "clear_lead") d <- d[!d$level, ]
  t <- logit_test(d$y)
  p <- published$specs[[i]]
  stopifnot(identical(unname(unlist(p$choices)), unname(unlist(ch))), t$n == p$n)
  near(t$effect, p$effect); near(t$ci_low, p$ci_low); near(t$ci_high, p$ci_high)
  near(t$p, p$p, 1e-12)
  out[[i]] <- data.frame(ch, k = t$k, n = t$n, effect = round(t$effect, 1),
                         ci_low = round(t$ci_low, 1), ci_high = round(t$ci_high, 1),
                         p = round(t$p, 4), yes = t$p < 0.05 && t$effect > 0)
}
specs <- do.call(rbind, out)
print(specs, row.names = FALSE)
cat(sprintf("yes on %d of %d paths\n", sum(specs$yes), nrow(specs)))

        seasons        table       ties  k  n effect ci_low ci_high      p
            all   first_half   official 44 63   19.8    7.5    29.9 0.0022
            all   first_half clear_lead 37 53   19.8    6.3    30.6 0.0051
            all winter_break   official 42 63   16.7    4.2    27.2 0.0095
            all winter_break clear_lead 38 53   21.7    8.2    32.2 0.0023
 without_bayern   first_half   official 19 35    4.3  -12.1    19.8 0.6125
 without_bayern   first_half clear_lead 16 29    5.2  -12.8    21.9 0.5781
 without_bayern winter_break   official 17 35   -1.4  -17.3    14.7 0.8658
 without_bayern winter_break clear_lead 16 28    7.1  -11.3    23.8 0.4513
   three_points   first_half   official 23 31   24.2    6.3    36.5 0.0101
   three_points   first_half clear_lead 21 27   27.8    8.6    39.7 0.0068
   three_points winter_break   official 23 31   24.2    6.3    36.5 0.0101
   three_points winter_break clear_lead 21 26   30.8   11.3    41.8 0.0039
   yes
  TRUE
  TRUE
  TR

yes on 8 of 12 paths


## Data and methods

- **Results 1963/64 to 2009/10:** [football.csv / deutschland](https://github.com/footballcsv/deutschland),
  CC0 1.0, an export of openfootball, accessed 17 November 2026.
- **Results 2010/11 to 2025/26:** [openfootball/deutschland](https://github.com/openfootball/deutschland),
  CC0 1.0. For 2010/11 to 2019/20 both sources have the same results on every matchday.
- **Cross-check:** halfway leaders from the German Wikipedia article "Herbstmeister", champions
  from the English Wikipedia "List of German football champions", both read on 17 November 2026;
  all 63 seasons agree.
- **Test:** logistic regression on a constant, Wald z-test of a share of one half, α = 0.05,
  two-sided; 95% confidence interval from the same standard error. No random numbers.
- **Pre-registration:** question, definitions, forks, test and the smallest relevant effect
  (10 percentage points) were fixed in the pitch (`pitch.md`) before any table was computed.

Notebook and code: © John Brüne, all rights reserved. Match data: CC0 1.0.